# Parkinson's Disease Detection — Colab Notebook

End-to-end pipeline: Roboflow download → EDA → leakage check → custom CNN / VGG16 / ResNet50 → evaluation.

**Before running:** `Runtime → Change runtime type → T4 GPU`, and add `ROBOFLOW_API_KEY` under Colab **Secrets** (key icon, left sidebar).

> ⚠️ Research / educational use only. Not a medical device; not for diagnosis.

## 1. Environment

In [ ]:
import tensorflow as tf
print("TensorFlow", tf.__version__)
print(tf.config.list_physical_devices("GPU") or "No GPU found -> Runtime > Change runtime type > T4 GPU")

In [ ]:
import os
PROJECT_DIR = "/content/parkinson-disease-detection"
for d in ("src", "notebooks", "outputs"):
    os.makedirs(f"{PROJECT_DIR}/{d}", exist_ok=True)
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

In [ ]:
!pip install -q roboflow imagehash

## 2. Project source files
These cells write the `src/` package (identical to the GitHub repo), so the notebook runs standalone.

In [ ]:
%%writefile requirements.txt
roboflow
tensorflow>=2.16
scikit-learn
matplotlib
seaborn
pandas
pillow
imagehash
numpy

In [ ]:
%%writefile src/__init__.py
"""Parkinson's disease detection from images (binary classification)."""
__version__ = "1.0.0"

In [ ]:
%%writefile src/config.py
"""Central configuration: paths, hyperparameters, seeds and helpers.

Every tunable value lives here so experiments are reproducible and the other
modules stay free of magic numbers.
"""
from __future__ import annotations

import os
import random
from pathlib import Path

import numpy as np

# --------------------------------------------------------------------------- #
# Paths
# --------------------------------------------------------------------------- #
PROJECT_ROOT = Path(__file__).resolve().parents[1]
DATA_DIR = PROJECT_ROOT / "data"                 # Roboflow export (git-ignored)
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODELS_DIR = OUTPUT_DIR / "models"               # .keras files (git-ignored)
CHECKPOINT_DIR = MODELS_DIR / "checkpoints"      # best-weights checkpoints
PLOTS_DIR = OUTPUT_DIR / "plots"
METRICS_DIR = OUTPUT_DIR / "metrics"

# --------------------------------------------------------------------------- #
# Roboflow
# --------------------------------------------------------------------------- #
ROBOFLOW_WORKSPACE = "anosh-goodwin"
ROBOFLOW_PROJECT = "parkinson-disease-detection-1eeyh"
ROBOFLOW_VERSION = 1            # change if the project has newer dataset versions
ROBOFLOW_FORMAT = "folder"      # classification export: train/ valid/ test/
ROBOFLOW_API_KEY_PLACEHOLDER = "YOUR_ROBOFLOW_API_KEY"

# --------------------------------------------------------------------------- #
# Data
# --------------------------------------------------------------------------- #
SPLITS = ("train", "valid", "test")
SPLIT_ALIASES = {
    "train": ("train",),
    "valid": ("valid", "val", "validation"),
    "test": ("test",),
}
IMG_SIZE = (224, 224)
IMG_SHAPE = (*IMG_SIZE, 3)
BATCH_SIZE = 16

NEGATIVE_CLASS = "healthy"      # label 0
POSITIVE_CLASS = "parkinson"    # label 1 (the class we care about detecting)
CLASS_NAMES = (NEGATIVE_CLASS, POSITIVE_CLASS)
POSITIVE_ALIASES = {"parkinson", "parkinsons", "parkinson's", "parkinson_disease", "pd"}
NEGATIVE_ALIASES = {"healthy", "normal", "control"}

# Leakage / duplicate detection (perceptual hash)
PHASH_SIZE = 16                 # 16x16 -> 256-bit hash
PHASH_THRESHOLD = 8             # max Hamming distance to call two images near-duplicates

# Augmentation (training split only). Disable flips if the images are orientation
# sensitive (e.g. handwriting / spiral direction).
AUG_HORIZONTAL_FLIP = True
AUG_ROTATION = 0.08             # fraction of 2*pi
AUG_ZOOM = 0.10
AUG_TRANSLATION = 0.05
AUG_CONTRAST = 0.15
AUG_BRIGHTNESS = 0.10

# --------------------------------------------------------------------------- #
# Training
# --------------------------------------------------------------------------- #
SEED = 42
MODEL_NAMES = ("custom_cnn", "vgg16", "resnet50")

EPOCHS_SCRATCH = 60             # custom CNN
EPOCHS_HEAD = 30                # transfer learning phase 1 (frozen backbone)
EPOCHS_FINE_TUNE = 30           # transfer learning phase 2 (partial unfreeze)

LR_SCRATCH = 1e-3
LR_HEAD = 1e-3
LR_FINE_TUNE = 1e-5

EARLY_STOP_PATIENCE = 8
EARLY_STOP_PATIENCE_SCRATCH = 12
REDUCE_LR_PATIENCE = 3
REDUCE_LR_FACTOR = 0.3
MIN_LR = 1e-7

DROPOUT = 0.5
L2_REG = 1e-4
UNFREEZE_LAYERS = {"vgg16": 4, "resnet50": 30}   # trailing layers unfrozen in phase 2

# --------------------------------------------------------------------------- #
# Evaluation
# --------------------------------------------------------------------------- #
DEFAULT_THRESHOLD = 0.5
N_BOOTSTRAP = 1000              # bootstrap resamples for confidence intervals


# --------------------------------------------------------------------------- #
# Helpers
# --------------------------------------------------------------------------- #
def ensure_dirs() -> None:
    """Create all output directories."""
    # DATA_DIR is deliberately NOT created here: Roboflow must create it itself.
    for d in (MODELS_DIR, CHECKPOINT_DIR, PLOTS_DIR, METRICS_DIR):
        d.mkdir(parents=True, exist_ok=True)


def set_global_seed(seed: int = SEED) -> None:
    """Seed Python, NumPy and TensorFlow for reproducible runs."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        import keras

        keras.utils.set_random_seed(seed)
    except Exception:  # pragma: no cover - TF not installed yet
        pass


def get_api_key() -> str:
    """Resolve the Roboflow API key without ever hard-coding it.

    Order: ``ROBOFLOW_API_KEY`` env var -> Colab Secrets (``userdata``).
    Raises if only the placeholder is available.
    """
    key = os.getenv("ROBOFLOW_API_KEY")
    if not key:
        try:
            from google.colab import userdata  # type: ignore

            key = userdata.get("ROBOFLOW_API_KEY")
        except Exception:
            key = None
    if not key or key == ROBOFLOW_API_KEY_PLACEHOLDER:
        raise RuntimeError(
            "Roboflow API key not found. Set the ROBOFLOW_API_KEY environment variable "
            "or add it to Colab Secrets (key icon in the left sidebar) as 'ROBOFLOW_API_KEY'."
        )
    return key

In [ ]:
%%writefile src/data_loader.py
"""Dataset download, EDA, leakage / duplicate checks and the tf.data pipeline."""
from __future__ import annotations

import hashlib
import itertools
import shutil
from pathlib import Path
from typing import Dict, Optional, Tuple

import imagehash
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from PIL import Image
from tensorflow import keras

from . import config

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
AUTOTUNE = tf.data.AUTOTUNE


# --------------------------------------------------------------------------- #
# 1. Download
# --------------------------------------------------------------------------- #
def _find_split_dir(root: Path, split: str) -> Optional[Path]:
    for name in config.SPLIT_ALIASES[split]:
        candidate = root / name
        if candidate.is_dir():
            return candidate
    return None


def download_dataset(
    api_key: Optional[str] = None,
    version: int = config.ROBOFLOW_VERSION,
    location: Path = config.DATA_DIR,
    force: bool = False,
) -> Path:
    """Download the Roboflow export in ``folder`` classification format.

    Skips the download if all splits already exist (unless ``force``).
    """
    location = Path(location)
    if not force and all(_find_split_dir(location, s) for s in config.SPLITS):
        print(f"[data] Dataset already present at {location}")
        return location
    if force and location.exists():
        shutil.rmtree(location)
    if location.exists() and not any(location.iterdir()):
        location.rmdir()  # an empty pre-existing folder makes the Roboflow extract unreliable

    from roboflow import Roboflow  # imported lazily: only needed for download

    rf = Roboflow(api_key=api_key or config.get_api_key())
    project = rf.workspace(config.ROBOFLOW_WORKSPACE).project(config.ROBOFLOW_PROJECT)
    dataset = project.version(version).download(config.ROBOFLOW_FORMAT, location=str(location))
    root = Path(dataset.location)
    print(f"[data] Downloaded to {root}")

    # Verify the export really contains the splits (also handles one level of nesting).
    for candidate in [root, *sorted(p for p in root.glob("*") if p.is_dir())]:
        if all(_find_split_dir(candidate, s) for s in config.SPLITS):
            return candidate
    found = sorted(str(p.relative_to(root)) for p in root.rglob("*") if p.is_dir())[:20]
    raise FileNotFoundError(
        f"Download finished but no train/valid/test folders were found in {root}. "
        f"Folders present: {found}. Check ROBOFLOW_WORKSPACE / ROBOFLOW_PROJECT / ROBOFLOW_VERSION."
    )


# --------------------------------------------------------------------------- #
# 2. Manifest
# --------------------------------------------------------------------------- #
def class_to_label(class_dir_name: str) -> int:
    name = class_dir_name.strip().lower()
    if name in config.POSITIVE_ALIASES:
        return 1
    if name in config.NEGATIVE_ALIASES:
        return 0
    raise ValueError(
        f"Unrecognised class folder '{class_dir_name}'. Expected one of "
        f"{sorted(config.NEGATIVE_ALIASES | config.POSITIVE_ALIASES)}; "
        "adjust the aliases in src/config.py."
    )


def _source_id(path: Path) -> str:
    """Roboflow names files ``<original>.rf.<hash>.jpg`` - recover the original name."""
    return path.name.split(".rf.")[0]


def build_manifest(root: Path) -> pd.DataFrame:
    """Scan the export and return one row per image."""
    root = Path(root)
    rows = []
    for split in config.SPLITS:
        split_dir = _find_split_dir(root, split)
        if split_dir is None:
            raise FileNotFoundError(f"Split '{split}' not found under {root}")
        for class_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
            label = class_to_label(class_dir.name)
            for f in sorted(class_dir.rglob("*")):
                if f.suffix.lower() in IMG_EXTS:
                    rows.append(
                        dict(
                            path=str(f),
                            split=split,
                            class_name=config.CLASS_NAMES[label],
                            label=label,
                            source_id=_source_id(f),
                        )
                    )
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f"No images found under {root}")
    return df


# --------------------------------------------------------------------------- #
# 3. EDA
# --------------------------------------------------------------------------- #
def run_eda(df: pd.DataFrame, show: bool = True) -> pd.DataFrame:
    """Class balance, image-size statistics and a sample grid. Returns df with sizes."""
    config.PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    counts = df.groupby(["split", "class_name"]).size().unstack(fill_value=0)
    counts = counts.reindex(list(config.SPLITS))
    print("\n[eda] Images per split / class\n", counts, sep="")
    print(f"[eda] Total images: {len(df)}")

    train_counts = counts.loc["train"]
    ratio = train_counts.max() / max(train_counts.min(), 1)
    print(f"[eda] Train imbalance ratio (majority/minority): {ratio:.2f}")

    sizes = []
    for p in df.path:
        with Image.open(p) as im:
            sizes.append((im.width, im.height, im.mode))
    df = df.copy()
    df[["width", "height", "mode"]] = pd.DataFrame(sizes, index=df.index)
    print("\n[eda] Image size statistics\n", df[["width", "height"]].describe().round(1), sep="")
    print("[eda] Colour modes:", df["mode"].value_counts().to_dict())

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    counts.plot(kind="bar", ax=axes[0], rot=0)
    axes[0].set_title("Images per split and class")
    axes[0].set_ylabel("count")
    sns.scatterplot(data=df, x="width", y="height", hue="class_name", ax=axes[1])
    axes[1].set_title("Original image dimensions")
    plt.tight_layout()
    fig.savefig(config.PLOTS_DIR / "eda_class_distribution.png", dpi=150)
    if show:
        plt.show()
    plt.close(fig)

    # Sample grid (train)
    train = df[df.split == "train"]
    n = 4
    fig, axes = plt.subplots(2, n, figsize=(3 * n, 6))
    for r, cls in enumerate(config.CLASS_NAMES):
        sample = train[train.class_name == cls].sample(
            min(n, (train.class_name == cls).sum()), random_state=config.SEED
        )
        for c in range(n):
            ax = axes[r, c]
            ax.axis("off")
            if c < len(sample):
                ax.imshow(Image.open(sample.iloc[c].path).convert("RGB"))
                ax.set_title(cls, fontsize=10)
    plt.tight_layout()
    fig.savefig(config.PLOTS_DIR / "eda_samples.png", dpi=150)
    if show:
        plt.show()
    plt.close(fig)
    return df


# --------------------------------------------------------------------------- #
# 4. Leakage / duplicate detection
# --------------------------------------------------------------------------- #
def _md5(path: str) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def find_duplicate_pairs(df: pd.DataFrame, threshold: int = config.PHASH_THRESHOLD) -> pd.DataFrame:
    """Find exact and near-duplicate image pairs (within and across splits).

    A pair is reported when any of the following holds:
      * byte-identical files (MD5)
      * perceptual-hash Hamming distance <= ``threshold``
      * same Roboflow source image appearing in different splits
    """
    df = df.reset_index(drop=True)
    md5s = [_md5(p) for p in df.path]
    phashes = []
    for p in df.path:
        with Image.open(p) as im:
            phashes.append(imagehash.phash(im.convert("RGB"), hash_size=config.PHASH_SIZE))

    rows = []
    for i, j in itertools.combinations(range(len(df)), 2):
        a, b = df.iloc[i], df.iloc[j]
        dist = int(phashes[i] - phashes[j])
        exact = md5s[i] == md5s[j]
        cross = a.split != b.split
        same_source = cross and a.source_id == b.source_id
        if exact or dist <= threshold or same_source:
            rows.append(
                dict(
                    path_a=a.path, split_a=a.split, label_a=a.label,
                    path_b=b.path, split_b=b.split, label_b=b.label,
                    hamming=dist, exact=exact, same_source=same_source,
                    cross_split=cross, label_conflict=a.label != b.label,
                )
            )
    return pd.DataFrame(rows)


def remove_leakage(df: pd.DataFrame, pairs: pd.DataFrame) -> Tuple[pd.DataFrame, list]:
    """Drop leaked images so valid/test stay pristine.

    * Cross-split pair -> the copy in the lower-priority split is removed
      (priority: test > valid > train).
    * Byte-identical copies inside one split -> keep the first, drop the rest.
    """
    if pairs.empty:
        return df.copy(), []
    rank = {"train": 0, "valid": 1, "test": 2}
    drop = set()
    for r in pairs.itertuples():
        if r.cross_split:
            drop.add(r.path_a if rank[r.split_a] < rank[r.split_b] else r.path_b)
        elif r.exact:
            drop.add(r.path_b)
    kept = df[~df.path.isin(drop)].reset_index(drop=True)
    return kept, sorted(drop)


def leakage_report(df: pd.DataFrame, remove: bool = True) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Run the duplicate scan, print a summary, optionally clean the manifest."""
    pairs = find_duplicate_pairs(df)
    config.METRICS_DIR.mkdir(parents=True, exist_ok=True)
    pairs.to_csv(config.METRICS_DIR / "duplicate_pairs.csv", index=False)

    if pairs.empty:
        print("\n[leakage] No exact/near-duplicate images found. Splits look clean.")
        return df, pairs

    cross = pairs[pairs.cross_split]
    print("\n[leakage] Duplicate scan summary")
    print(f"  total flagged pairs     : {len(pairs)}")
    print(f"  cross-split (LEAKAGE)   : {len(cross)}")
    print(f"  exact byte duplicates   : {int(pairs.exact.sum())}")
    print(f"  same-source cross-split : {int(pairs.same_source.sum())}")
    print(f"  conflicting labels      : {int(pairs.label_conflict.sum())}")
    if pairs.label_conflict.any():
        print("  WARNING: near-identical images carry different labels -> inspect duplicate_pairs.csv")

    if not remove:
        print("  remove_leaked=False -> manifest left untouched.")
        return df, pairs

    cleaned, dropped = remove_leakage(df, pairs)
    print(f"  removed {len(dropped)} image(s) from train/valid; {len(cleaned)} images remain")
    return cleaned, pairs


# --------------------------------------------------------------------------- #
# 5. tf.data pipeline
# --------------------------------------------------------------------------- #
def build_augmenter() -> keras.Sequential:
    """Mild, medically-plausible augmentation applied to training batches only."""
    layers = []
    if config.AUG_HORIZONTAL_FLIP:
        layers.append(keras.layers.RandomFlip("horizontal"))
    layers += [
        keras.layers.RandomRotation(config.AUG_ROTATION, fill_mode="reflect"),
        keras.layers.RandomZoom(config.AUG_ZOOM, fill_mode="reflect"),
        keras.layers.RandomTranslation(config.AUG_TRANSLATION, config.AUG_TRANSLATION, fill_mode="reflect"),
        keras.layers.RandomContrast(config.AUG_CONTRAST),
        keras.layers.RandomBrightness(config.AUG_BRIGHTNESS, value_range=(0, 255)),
    ]
    return keras.Sequential(layers, name="augmentation")


def _decode(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, config.IMG_SIZE, antialias=True)
    return tf.cast(img, tf.float32), tf.cast(label, tf.float32)


def make_dataset(
    df: pd.DataFrame,
    training: bool,
    batch_size: int = config.BATCH_SIZE,
    augment: bool = True,
) -> tf.data.Dataset:
    """Build a dataset yielding ``(image[0-255 float32], label)``.

    Pixel scaling / backbone-specific preprocessing happens *inside* each model so
    saved ``.keras`` files accept raw images. Evaluation datasets keep the
    manifest order so predictions can be mapped back to file paths.
    """
    paths = df.path.astype(str).tolist()
    labels = df.label.astype("float32").tolist()
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(_decode, num_parallel_calls=AUTOTUNE).cache()
    if training:
        ds = ds.shuffle(len(df), seed=config.SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size)
    if training and augment:
        aug = build_augmenter()
        ds = ds.map(lambda x, y: (aug(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)


def build_datasets(splits: Dict[str, pd.DataFrame]) -> Dict[str, tf.data.Dataset]:
    return {
        "train": make_dataset(splits["train"], training=True),
        "valid": make_dataset(splits["valid"], training=False),
        "test": make_dataset(splits["test"], training=False),
    }


# --------------------------------------------------------------------------- #
# 6. One-call preparation
# --------------------------------------------------------------------------- #
def prepare_data(
    api_key: Optional[str] = None,
    remove_leaked: bool = True,
    do_eda: bool = True,
    show: bool = True,
) -> dict:
    """Download -> manifest -> EDA -> leakage check -> per-split DataFrames."""
    config.ensure_dirs()
    root = download_dataset(api_key=api_key)
    manifest = build_manifest(root)
    if do_eda:
        manifest = run_eda(manifest, show=show)
    manifest, pairs = leakage_report(manifest, remove=remove_leaked)
    splits = {s: manifest[manifest.split == s].reset_index(drop=True) for s in config.SPLITS}
    for s, d in splits.items():
        if d.label.nunique() < 2:
            raise RuntimeError(f"Split '{s}' contains a single class after cleaning - check the dataset.")
    manifest.to_csv(config.METRICS_DIR / "manifest_clean.csv", index=False)
    return {"root": root, "manifest": manifest, "splits": splits, "duplicates": pairs}

In [ ]:
%%writefile src/models.py
"""Model architectures: a small custom CNN plus VGG16 / ResNet50 transfer learners.

All models take raw ``[0, 255]`` RGB images of shape ``config.IMG_SHAPE`` and output a
single sigmoid probability ``P(parkinson)``. Preprocessing lives *inside* the model so
saved ``.keras`` files can be used directly for inference.
"""
from __future__ import annotations

from typing import Callable, Dict, Optional, Tuple

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

from . import config


@keras.saving.register_keras_serializable(package="parkinson")
class CaffePreprocess(layers.Layer):
    """RGB -> BGR and ImageNet mean subtraction (what VGG16 / ResNet50 weights expect)."""

    def call(self, x):
        mean = tf.constant([103.939, 116.779, 123.68], dtype=x.dtype)
        return tf.reverse(x, axis=[-1]) - mean


def _head(x, dense_units: int = 128):
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(config.DROPOUT)(x)
    x = layers.Dense(dense_units, activation="relu", kernel_regularizer=regularizers.l2(config.L2_REG))(x)
    x = layers.Dropout(config.DROPOUT / 2)(x)
    return layers.Dense(1, activation="sigmoid", name="p_parkinson")(x)


# --------------------------------------------------------------------------- #
# Custom CNN (trained from scratch, deliberately small for ~190 images)
# --------------------------------------------------------------------------- #
def build_custom_cnn() -> keras.Model:
    inputs = keras.Input(shape=config.IMG_SHAPE)
    x = layers.Rescaling(1.0 / 255)(inputs)
    for filters in (32, 64, 128, 128):
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False,
                          kernel_regularizer=regularizers.l2(config.L2_REG))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.SpatialDropout2D(0.1)(x)
    outputs = _head(x, dense_units=64)
    return keras.Model(inputs, outputs, name="custom_cnn")


# --------------------------------------------------------------------------- #
# Transfer learning
# --------------------------------------------------------------------------- #
def _build_transfer(backbone_fn: Callable, base_name: str, model_name: str,
                    weights: Optional[str] = "imagenet") -> keras.Model:
    base = backbone_fn(include_top=False, weights=weights, input_shape=config.IMG_SHAPE, name=base_name)
    base.trainable = False  # phase 1: train only the new head
    inputs = keras.Input(shape=config.IMG_SHAPE)
    x = CaffePreprocess(name="caffe_preprocess")(inputs)
    x = base(x, training=False)  # keep BatchNorm in inference mode, even when fine-tuning
    outputs = _head(x)
    return keras.Model(inputs, outputs, name=model_name)


def build_vgg16(weights: Optional[str] = "imagenet") -> keras.Model:
    return _build_transfer(keras.applications.VGG16, "vgg16_base", "vgg16", weights)


def build_resnet50(weights: Optional[str] = "imagenet") -> keras.Model:
    return _build_transfer(keras.applications.ResNet50, "resnet50_base", "resnet50", weights)


MODEL_REGISTRY: Dict[str, Callable[[], keras.Model]] = {
    "custom_cnn": build_custom_cnn,
    "vgg16": build_vgg16,
    "resnet50": build_resnet50,
}
BASE_LAYER_NAMES = {"vgg16": "vgg16_base", "resnet50": "resnet50_base"}


def get_model(name: str) -> keras.Model:
    if name not in MODEL_REGISTRY:
        raise KeyError(f"Unknown model '{name}'. Choose from {list(MODEL_REGISTRY)}")
    return MODEL_REGISTRY[name]()


def is_transfer_model(name: str) -> bool:
    return name in BASE_LAYER_NAMES


def unfreeze_top_layers(model: keras.Model, name: str, n_layers: Optional[int] = None) -> int:
    """Phase-2 helper: unfreeze the last ``n_layers`` of the backbone.

    BatchNorm layers stay frozen (standard practice for tiny datasets). The model
    must be re-compiled afterwards. Returns the number of trainable backbone layers.
    """
    n_layers = n_layers or config.UNFREEZE_LAYERS[name]
    base = model.get_layer(BASE_LAYER_NAMES[name])
    base.trainable = True
    for layer in base.layers[:-n_layers]:
        layer.trainable = False
    for layer in base.layers[-n_layers:]:
        layer.trainable = not isinstance(layer, layers.BatchNormalization)
    return sum(layer.trainable for layer in base.layers)

In [ ]:
%%writefile src/train.py
"""Training loops, callbacks, class weights and two-phase fine-tuning.

Run end-to-end from the repo root::

    python -m src.train --models custom_cnn vgg16 resnet50
"""
from __future__ import annotations

import argparse
import gc
from typing import Dict, Iterable, Tuple

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
from tensorflow import keras

from . import config, data_loader, models


def configure_gpu() -> None:
    """Enable memory growth and report available devices."""
    gpus = tf.config.list_physical_devices("GPU")
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError:
            pass
    print(f"[gpu] TensorFlow {tf.__version__} | GPUs: {[g.name for g in gpus] or 'none (CPU only)'}")


def compute_class_weights(train_df: pd.DataFrame) -> Dict[int, float]:
    """Balanced class weights computed from the *training* split only."""
    classes = np.array([0, 1])
    weights = compute_class_weight("balanced", classes=classes, y=train_df.label.values)
    out = {int(c): float(w) for c, w in zip(classes, weights)}
    print(f"[train] class weights: {out}")
    return out


def compile_model(model: keras.Model, lr: float) -> None:
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.AUC(name="auc"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
        ],
    )


def _callbacks(name: str, tag: str, patience: int):
    config.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    config.METRICS_DIR.mkdir(parents=True, exist_ok=True)
    ckpt = str(config.CHECKPOINT_DIR / f"{name}_{tag}.weights.h5")
    cbs = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=config.REDUCE_LR_FACTOR,
            patience=config.REDUCE_LR_PATIENCE, min_lr=config.MIN_LR, verbose=1,
        ),
        # Best weights are saved explicitly and reloaded after fit(): robust across Keras versions.
        keras.callbacks.ModelCheckpoint(
            ckpt, monitor="val_loss", save_best_only=True, save_weights_only=True
        ),
    ]
    return cbs, ckpt


def _fit_phase(model, train_ds, val_ds, epochs, lr, class_weight, name, tag, patience) -> Tuple[pd.DataFrame, float]:
    compile_model(model, lr)
    cbs, ckpt = _callbacks(name, tag, patience)
    hist = model.fit(
        train_ds, validation_data=val_ds, epochs=epochs,
        class_weight=class_weight, callbacks=cbs, verbose=2,
    )
    model.load_weights(ckpt)  # guarantee we continue from / return the best epoch
    df = pd.DataFrame(hist.history)
    df["epoch"] = np.arange(1, len(df) + 1)
    df["phase"] = tag
    return df, float(df["val_loss"].min())


def train_model(
    name: str,
    train_ds: tf.data.Dataset,
    val_ds: tf.data.Dataset,
    class_weight: Dict[int, float],
) -> Tuple[keras.Model, pd.DataFrame]:
    """Train one architecture.

    * ``custom_cnn``: single phase from scratch.
    * ``vgg16`` / ``resnet50``: phase 1 trains the new head with the backbone frozen;
      phase 2 unfreezes the top backbone layers at a much lower learning rate. The
      better phase (by validation loss) is kept.
    """
    config.set_global_seed()
    model = models.get_model(name)
    print(f"\n{'=' * 70}\nTraining {name}  ({model.count_params():,} params)\n{'=' * 70}")

    if not models.is_transfer_model(name):
        hist, _ = _fit_phase(model, train_ds, val_ds, config.EPOCHS_SCRATCH, config.LR_SCRATCH,
                             class_weight, name, "scratch", config.EARLY_STOP_PATIENCE_SCRATCH)
        history = hist
    else:
        print("[train] Phase 1: frozen backbone, training classification head")
        h1, loss1 = _fit_phase(model, train_ds, val_ds, config.EPOCHS_HEAD, config.LR_HEAD,
                               class_weight, name, "head", config.EARLY_STOP_PATIENCE)

        n = models.unfreeze_top_layers(model, name)
        print(f"[train] Phase 2: fine-tuning top {n} backbone layers (lr={config.LR_FINE_TUNE})")
        h2, loss2 = _fit_phase(model, train_ds, val_ds, config.EPOCHS_FINE_TUNE, config.LR_FINE_TUNE,
                               class_weight, name, "fine_tune", config.EARLY_STOP_PATIENCE)
        if loss2 > loss1:  # fine-tuning did not help -> fall back to the phase-1 weights
            print(f"[train] Fine-tune val_loss {loss2:.4f} > head val_loss {loss1:.4f}; reverting to phase 1")
            model.load_weights(str(config.CHECKPOINT_DIR / f"{name}_head.weights.h5"))
        history = pd.concat([h1, h2], ignore_index=True)

    config.MODELS_DIR.mkdir(parents=True, exist_ok=True)
    model.save(config.MODELS_DIR / f"{name}.keras")
    history.to_csv(config.METRICS_DIR / f"history_{name}.csv", index=False)
    return model, history


def train_all(
    model_names: Iterable[str],
    train_ds: tf.data.Dataset,
    val_ds: tf.data.Dataset,
    class_weight: Dict[int, float],
) -> Tuple[Dict[str, keras.Model], Dict[str, pd.DataFrame]]:
    trained, histories = {}, {}
    for name in model_names:
        trained[name], histories[name] = train_model(name, train_ds, val_ds, class_weight)
        gc.collect()
    return trained, histories


def main() -> None:
    parser = argparse.ArgumentParser(description="Train Parkinson's detection models")
    parser.add_argument("--models", nargs="+", default=list(config.MODEL_NAMES), choices=list(models.MODEL_REGISTRY))
    parser.add_argument("--keep-leaked", action="store_true", help="do not remove cross-split duplicates")
    parser.add_argument("--no-eda", action="store_true")
    args = parser.parse_args()

    configure_gpu()
    config.set_global_seed()
    data = data_loader.prepare_data(remove_leaked=not args.keep_leaked, do_eda=not args.no_eda, show=False)
    ds = data_loader.build_datasets(data["splits"])
    cw = compute_class_weights(data["splits"]["train"])
    train_all(args.models, ds["train"], ds["valid"], cw)
    print("\nDone. Run `python -m src.evaluate` to score the saved models on the test set.")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/evaluate.py
"""Metrics, comparison table, plots, confusion matrices, ROC curves and error analysis.

Model selection and threshold tuning use the *validation* split only; the test split is
touched exactly once for final reporting.

Run from the repo root to score saved models::

    python -m src.evaluate
"""
from __future__ import annotations

import argparse
from typing import Dict, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score, brier_score_loss,
    confusion_matrix, f1_score, matthews_corrcoef, precision_score, recall_score,
    roc_auc_score, roc_curve,
)
from tensorflow import keras

from . import config, data_loader, models  # noqa: F401  (models import registers custom layers)


# --------------------------------------------------------------------------- #
# Predictions & metrics
# --------------------------------------------------------------------------- #
def predict_probs(model: keras.Model, ds) -> np.ndarray:
    """Probabilities in dataset order (evaluation datasets are never shuffled)."""
    return model.predict(ds, verbose=0).ravel()


def youden_threshold(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    """Threshold maximising sensitivity + specificity - 1 (chosen on validation data)."""
    if len(np.unique(y_true)) < 2:
        return config.DEFAULT_THRESHOLD
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    best = int(np.argmax(tpr - fpr))
    return float(np.clip(thr[best], 0.0, 1.0))


def compute_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float) -> dict:
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    two_classes = len(np.unique(y_true)) == 2
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),            # sensitivity
        "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_prob) if two_classes else np.nan,
        "pr_auc": average_precision_score(y_true, y_prob) if two_classes else np.nan,
        "brier": brier_score_loss(y_true, y_prob),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def bootstrap_ci(y_true, y_prob, fn, n: int = config.N_BOOTSTRAP, seed: int = config.SEED):
    """95% percentile bootstrap CI - essential with a test set of only a few dozen images."""
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n):
        idx = rng.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[idx])) < 2:
            continue
        vals.append(fn(y_true[idx], y_prob[idx]))
    if not vals:
        return (np.nan, np.nan)
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return float(lo), float(hi)


def evaluate_models(trained: Dict[str, keras.Model], splits: Dict[str, pd.DataFrame], datasets: dict):
    """Score every model on valid + test. Returns (comparison_table, predictions)."""
    config.METRICS_DIR.mkdir(parents=True, exist_ok=True)
    y_val, y_test = splits["valid"].label.values, splits["test"].label.values
    rows, preds = [], {}
    for name, model in trained.items():
        p_val, p_test = predict_probs(model, datasets["valid"]), predict_probs(model, datasets["test"])
        thr_tuned = youden_threshold(y_val, p_val)
        preds[name] = {"val": p_val, "test": p_test, "thr_tuned": thr_tuned}
        val_m = compute_metrics(y_val, p_val, config.DEFAULT_THRESHOLD)
        for mode, thr in (("0.5", config.DEFAULT_THRESHOLD), ("val_tuned", thr_tuned)):
            m = compute_metrics(y_test, p_test, thr)
            acc_ci = bootstrap_ci(y_test, p_test, lambda y, p, t=thr: accuracy_score(y, (p >= t).astype(int)))
            auc_ci = bootstrap_ci(y_test, p_test, roc_auc_score)
            rows.append({
                "model": name, "threshold_mode": mode, "threshold": round(thr, 4),
                **m,
                "accuracy_ci95": f"[{acc_ci[0]:.3f}, {acc_ci[1]:.3f}]",
                "auc_ci95": f"[{auc_ci[0]:.3f}, {auc_ci[1]:.3f}]",
                "val_auc": val_m["roc_auc"], "val_accuracy": val_m["accuracy"],
            })
    table = pd.DataFrame(rows)
    table.to_csv(config.METRICS_DIR / "model_comparison.csv", index=False)
    return table, preds


def select_best_model(table: pd.DataFrame) -> str:
    """Pick the winner from *validation* AUC (ties -> validation accuracy), never from test."""
    t = table.drop_duplicates("model").sort_values(["val_auc", "val_accuracy"], ascending=False)
    return str(t.iloc[0]["model"])


def summary_view(table: pd.DataFrame, mode: str = "0.5") -> pd.DataFrame:
    cols = ["model", "accuracy", "balanced_accuracy", "precision", "recall", "specificity",
            "f1", "roc_auc", "pr_auc", "mcc", "accuracy_ci95", "auc_ci95", "val_auc"]
    return (table[table.threshold_mode == mode][cols]
            .sort_values("val_auc", ascending=False).round(3).reset_index(drop=True))


# --------------------------------------------------------------------------- #
# Plots
# --------------------------------------------------------------------------- #
def _finish(fig, filename: str, show: bool = True) -> None:
    config.PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(config.PLOTS_DIR / filename, dpi=150, bbox_inches="tight")
    if show:
        plt.show()
    plt.close(fig)


def plot_training_curves(histories: Dict[str, pd.DataFrame], show: bool = True) -> None:
    fig, axes = plt.subplots(len(histories), 3, figsize=(16, 3.8 * len(histories)), squeeze=False)
    for r, (name, h) in enumerate(histories.items()):
        x = np.arange(1, len(h) + 1)
        for c, (metric, title) in enumerate((("loss", "Loss"), ("accuracy", "Accuracy"), ("auc", "AUC"))):
            ax = axes[r, c]
            ax.plot(x, h[metric], label="train")
            ax.plot(x, h[f"val_{metric}"], label="valid")
            changes = np.where(h["phase"].values[1:] != h["phase"].values[:-1])[0]
            for ch in changes:
                ax.axvline(ch + 1.5, color="grey", ls="--", lw=1)
                ax.text(ch + 1.6, ax.get_ylim()[0], " fine-tune", fontsize=8, va="bottom")
            ax.set_title(f"{name} - {title}")
            ax.set_xlabel("epoch")
            ax.legend()
    _finish(fig, "training_curves.png", show)


def plot_confusion_matrices(preds: dict, splits: dict, show: bool = True) -> None:
    y = splits["test"].label.values
    names = list(preds)
    fig, axes = plt.subplots(2, len(names), figsize=(4.2 * len(names), 7.5), squeeze=False)
    for c, name in enumerate(names):
        for r, (mode, thr) in enumerate((("0.5", config.DEFAULT_THRESHOLD), ("val-tuned", preds[name]["thr_tuned"]))):
            cm = confusion_matrix(y, (preds[name]["test"] >= thr).astype(int), labels=[0, 1])
            sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[r, c],
                        xticklabels=config.CLASS_NAMES, yticklabels=config.CLASS_NAMES)
            axes[r, c].set_title(f"{name}\nthreshold={thr:.2f} ({mode})")
            axes[r, c].set_xlabel("predicted")
            axes[r, c].set_ylabel("true")
    _finish(fig, "confusion_matrices.png", show)


def plot_roc_curves(preds: dict, splits: dict, show: bool = True) -> None:
    y = splits["test"].label.values
    fig, ax = plt.subplots(figsize=(6.5, 6))
    for name, p in preds.items():
        fpr, tpr, _ = roc_curve(y, p["test"])
        ax.plot(fpr, tpr, lw=2, label=f"{name} (AUC={roc_auc_score(y, p['test']):.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="chance")
    ax.set(xlabel="False positive rate", ylabel="True positive rate (sensitivity)", title="ROC curves - test set")
    ax.legend(loc="lower right")
    _finish(fig, "roc_curves.png", show)


def plot_metric_comparison(table: pd.DataFrame, mode: str = "0.5", show: bool = True) -> None:
    metrics = ["accuracy", "precision", "recall", "specificity", "f1", "roc_auc"]
    long = table[table.threshold_mode == mode].melt(id_vars="model", value_vars=metrics,
                                                    var_name="metric", value_name="score")
    fig, ax = plt.subplots(figsize=(10, 4.5))
    sns.barplot(data=long, x="metric", y="score", hue="model", ax=ax)
    ax.set_ylim(0, 1.05)
    ax.set_title(f"Test-set metrics (threshold mode: {mode})")
    _finish(fig, "metric_comparison.png", show)


# --------------------------------------------------------------------------- #
# Error analysis
# --------------------------------------------------------------------------- #
def error_analysis(name: str, test_df: pd.DataFrame, probs: np.ndarray,
                   threshold: float = config.DEFAULT_THRESHOLD, max_images: int = 12,
                   show: bool = True) -> pd.DataFrame:
    """Save per-image predictions and plot misclassified images (FN are the costly errors)."""
    out = test_df[["path", "class_name", "label"]].copy()
    out["p_parkinson"] = probs
    out["pred"] = (probs >= threshold).astype(int)
    out["correct"] = out.pred == out.label
    out["error_type"] = np.where(out.correct, "ok", np.where(out.label == 1, "false_negative", "false_positive"))
    out["confidence"] = np.where(out.pred == 1, probs, 1 - probs)
    config.METRICS_DIR.mkdir(parents=True, exist_ok=True)
    out.to_csv(config.METRICS_DIR / f"predictions_{name}.csv", index=False)

    errs = out[~out.correct].sort_values("confidence", ascending=False)
    print(f"[errors] {name}: {len(errs)} / {len(out)} misclassified "
          f"(FN={int((errs.error_type == 'false_negative').sum())}, FP={int((errs.error_type == 'false_positive').sum())})")
    if errs.empty:
        print("[errors] No misclassifications on the test set.")
        return out

    errs = errs.head(max_images)
    cols = min(4, len(errs))
    rows = int(np.ceil(len(errs) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(3.4 * cols, 3.6 * rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, (_, r) in zip(axes.ravel(), errs.iterrows()):
        ax.imshow(Image.open(r.path).convert("RGB").resize(config.IMG_SIZE))
        ax.set_title(f"true: {r.class_name}\n{r.error_type} (p={r.p_parkinson:.2f})", fontsize=9, color="crimson")
    fig.suptitle(f"{name}: most confident misclassifications", y=1.02)
    _finish(fig, f"error_analysis_{name}.png", show)
    return out


# --------------------------------------------------------------------------- #
# CLI: evaluate saved models
# --------------------------------------------------------------------------- #
def main() -> None:
    parser = argparse.ArgumentParser(description="Evaluate saved models on the test split")
    parser.add_argument("--models", nargs="+", default=list(config.MODEL_NAMES))
    args = parser.parse_args()

    config.set_global_seed()
    data = data_loader.prepare_data(do_eda=False, show=False)
    ds = data_loader.build_datasets(data["splits"])
    trained = {n: keras.models.load_model(config.MODELS_DIR / f"{n}.keras") for n in args.models}

    table, preds = evaluate_models(trained, data["splits"], ds)
    print(summary_view(table, "0.5").to_string(index=False))
    print("\nVal-tuned threshold:\n", summary_view(table, "val_tuned").to_string(index=False))

    plot_roc_curves(preds, data["splits"], show=False)
    plot_confusion_matrices(preds, data["splits"], show=False)
    plot_metric_comparison(table, show=False)
    best = select_best_model(table)
    print(f"\nBest model by validation AUC: {best}")
    error_analysis(best, data["splits"]["test"], preds[best]["test"], show=False)


if __name__ == "__main__":
    main()

## 3. Imports & API key
The key is read from the environment / Colab Secrets and never written to disk.

In [ ]:
import os, sys, getpass, importlib
if not os.getenv("ROBOFLOW_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
    except Exception:
        os.environ["ROBOFLOW_API_KEY"] = getpass.getpass("Roboflow API key: ")

sys.path.insert(0, PROJECT_DIR)
importlib.invalidate_caches()
from src import config, data_loader, models, train, evaluate

config.ensure_dirs()
config.set_global_seed()
train.configure_gpu()

## 4. Download, EDA and leakage check
Set `REMOVE_LEAKED = False` to only report duplicates without removing them.

In [ ]:
REMOVE_LEAKED = True
data = data_loader.prepare_data(remove_leaked=REMOVE_LEAKED)
splits = data["splits"]
data["duplicates"].head(20)

## 5. Input pipeline & class weights

In [ ]:
ds = data_loader.build_datasets(splits)
class_weight = train.compute_class_weights(splits["train"])
for name, df in splits.items():
    print(name, len(df), df.class_name.value_counts().to_dict())

## 6. Train
Custom CNN trains from scratch; VGG16 / ResNet50 use two-phase fine-tuning (frozen head → partial unfreeze).

In [ ]:
MODEL_NAMES = ["custom_cnn", "vgg16", "resnet50"]
trained, histories = train.train_all(MODEL_NAMES, ds["train"], ds["valid"], class_weight)

In [ ]:
evaluate.plot_training_curves(histories)

## 7. Evaluate
The test set is used once for reporting; model selection and the tuned threshold come from the validation set only.

In [ ]:
table, preds = evaluate.evaluate_models(trained, splits, ds)
print("Threshold = 0.5")
display(evaluate.summary_view(table, "0.5"))
print("Threshold tuned on validation set (Youden's J)")
display(evaluate.summary_view(table, "val_tuned"))

In [ ]:
evaluate.plot_roc_curves(preds, splits)
evaluate.plot_confusion_matrices(preds, splits)
evaluate.plot_metric_comparison(table)

## 8. Error analysis

In [ ]:
best = evaluate.select_best_model(table)
print("Best model by validation AUC:", best)
errors = evaluate.error_analysis(best, splits["test"], preds[best]["test"])
errors[~errors.correct]

## 9. Download outputs

In [ ]:
import shutil
shutil.make_archive("/content/parkinson_outputs", "zip", config.OUTPUT_DIR)
try:
    from google.colab import files
    files.download("/content/parkinson_outputs.zip")
except Exception:
    print("Saved to /content/parkinson_outputs.zip")